In [15]:
# Import the packages needed inside the function.
import wrds
import pandas as pd


def build_financial_ratio_report(username, company_dict, start_year):
    """
    Download financial ratio data from WRDS and return four pivot tables.

    Parameters:
        username (str): WRDS username
        company_dict (dict): mapping from stock code to company name
        start_year (int): keep data from this year onward

    Returns:
        dict: a dictionary containing the cleaned data and four markdown tables
    """

    # --- Connect to WRDS ---
    # A new connection is created inside the function so it is self-contained.
    conn = wrds.Connection(wrds_username=username)

    # --- Build the stock list for the SQL IN clause ---
    # .keys() returns all the dictionary keys (the stock codes).
    # tuple() converts the list of keys into a tuple
    # SQL's IN clause expects this format: WHERE stkcd IN 
    stock_list = tuple(company_dict.keys())

    # --- Build the SQL query ---
    # This query calculates four financial ratios directly inside SQL using AS.
    # The five ratios:
    #   roe          = Net Profit / Total Equity  (Return on Equity)
    #   roa          = Net Profit / Total Assets  (Return on Assets)
    #   profitmargin = Net Profit / Total Revenue (how much profit per yuan of revenue)
    #   turnover     = Total Revenue / Total Assets (how efficiently assets generate revenue)
    #   leverage     = Total Assets / Total Equity  (how much of assets are funded by debt)
    sql = f"""
    SELECT
    stkcd,
    accper,
    b002000000 / NULLIF(a003000000, 0) AS roe,
    b002000000 / NULLIF(a001000000, 0) AS roa,
    b002000000 / NULLIF(b001101000, 0) AS profitmargin,
    b001101000 / NULLIF(a001000000, 0) AS turnover,
    a001000000 / NULLIF(a003000000, 0) AS leverage
FROM csmar.wrds_csmar_financial_master
WHERE stkcd IN ('601857', '600028')
  AND typrep = 'A'
    """

    # --- Run the query and convert accper to datetime ---
    data = conn.raw_sql(sql, date_cols=["accper"])

    # Always close the connection after downloading data.
    conn.close()

    # --- Filter to December (year-end) observations only ---
    # .dt.month == 12 keeps only rows where accper falls in December.
    # .copy() avoids a pandas warning when we modify the filtered DataFrame.
    df = data[data["accper"].dt.month == 12].copy()

    # Extract the year from accper and store it as a new 'year' column.
    df["year"] = df["accper"].dt.year

    # Keep only data from start_year onward.
    df = df[df["year"] >= start_year].copy()

    # Drop accper now that we have the 'year' column — it is no longer needed.
    df = df.drop(columns=["accper"])

    # --- Map stock codes to readable company names ---
    # .map(company_dict) looks up each value in the 'stkcd' column in the dictionary
    # and replaces it with the corresponding company name.
    # For example: '000333' → 'Midea'
    df["Company"] = df["stkcd"].map(company_dict)

    # --- Create pivot tables ---
    # pivot() reshapes a DataFrame from 'long' format to 'wide' format.
    # pivot() parameters:
    #   index   = the column whose values become the row labels (here: Company)
    #   columns = the column whose values become the new column headers (here: year)
    #   values  = the column containing the numbers to fill the table (here: roe)
    #
    # After pivot(), we apply formatting:
    #   .mul(100)  → multiply by 100 to convert decimal to percentage 
    #   .round(1)  → round to 1 decimal place 
    #   .astype(str) + '%' → convert to string and add a '%' sign 
    #
    # .rename_axis(columns=None) removes the label 'year' above the column headers.
    # .rename_axis('ROE', axis='index') renames the row index label to 'ROE'.

    # ROE pivot table
    roe_pivot = (
        df.pivot(index="Company", columns="year", values="roe")
        .mul(100)
        .round(1)
        .astype(str)
        + "%"
    )
    roe_pivot = roe_pivot.rename_axis(columns=None)
    roe_pivot = roe_pivot.rename_axis("ROE", axis="index")
    
    # ROA pivot table
    roa_pivot = (
        df.pivot(index="Company", columns="year", values="roa")
        .mul(100)
        .round(1)
        .astype(str)
        + "%"
    )
    roa_pivot = roa_pivot.rename_axis(columns=None)
    roa_pivot = roa_pivot.rename_axis("ROA", axis="index")

    # Profit Margin pivot table (same structure as ROE)
    pm_pivot = (
        df.pivot(index="Company", columns="year", values="profitmargin")
        .mul(100)
        .round(1)
        .astype(str)
        + "%"
    )
    pm_pivot = pm_pivot.rename_axis(columns=None)
    pm_pivot = pm_pivot.rename_axis("Profit Margin", axis="index")

    # Asset Turnover pivot table (same structure)
    to_pivot = (
        df.pivot(index="Company", columns="year", values="turnover")
        .mul(100)
        .round(1)
        .astype(str)
        + "%"
    )
    to_pivot = to_pivot.rename_axis(columns=None)
    to_pivot = to_pivot.rename_axis("Turnover", axis="index")

    # Leverage pivot table
    # Leverage is a ratio (e.g. 2.90), not a percentage, so we skip .mul(100) and '%'.
    lev_pivot = (
        df.pivot(index="Company", columns="year", values="leverage")
        .round(2)
    )
    lev_pivot = lev_pivot.rename_axis(columns=None)
    lev_pivot = lev_pivot.rename_axis("Leverage", axis="index")

    # --- Return all outputs packed into one dictionary ---
    # Returning a dictionary lets the caller access any specific piece they need
    # using result['roe_pivot'], result['pm_markdown'], etc.
    return {
        "raw_data":     df,           # the cleaned long-format DataFrame
        "roe_pivot":    roe_pivot,     # ROE as a wide pivot table
        "roa_pivot":    roa_pivot,     # ROA as a wide pivot table
        "pm_pivot":     pm_pivot,      # Profit Margin pivot table
        "to_pivot":     to_pivot,      # Turnover pivot table
        "lev_pivot":    lev_pivot,     # Leverage pivot table
        "roe_markdown": roe_pivot.to_markdown(),  # ROE as a Markdown string
        "pm_markdown":  pm_pivot.to_markdown(),
        "to_markdown":  to_pivot.to_markdown(),
        "lev_markdown": lev_pivot.to_markdown(),
    }


In [18]:
# --- Define the inputs we will pass to the function ---

# Replace with your own WRDS username.
username = 'tsingli'

# company_dict maps each company name in WRDS (key)
# to the readable company label you want to display (value).
company_dict = {
    '601857': 'PetrolChina ',
    '600028': 'Sinopec'
}

# Only keep financial data from this year onward (inclusive).
start_year = 2021


In [19]:
# --- Call the function with the parameters defined above ---
result = build_financial_ratio_report(
    username=username,
    company_dict=company_dict,
    start_year=start_year
)

Loading library list...
Done


In [5]:
# Confirm that 'result' is a Python dictionary.
# type() returns the data type of any Python object.
type(result)

dict

In [6]:
# List all the keys (output names) available in the result dictionary.
# This is a list comprehension: it creates a list by iterating over the keys.
# It is equivalent to: list(result.keys())
[key for key in result]

['raw_data',
 'roe_pivot',
 'roa_pivot',
 'pm_pivot',
 'to_pivot',
 'lev_pivot',
 'roe_markdown',
 'pm_markdown',
 'to_markdown',
 'lev_markdown']

In [20]:
# Access the 'raw_data' item from the result dictionary.
# result['raw_data'] is the cleaned long-format DataFrame
# with columns: stkcd, roe, roa,  profitmargin, turnover, leverage, year, Company.
# .head() shows only the first 5 rows as a quick preview.
result["raw_data"]

,stkcd,roe,roa,profitmargin,turnover,leverage,year,Company
102,600028,0.092823,0.045007,0.031023,1.450775,2.062413,2021,Sinopec
107,600028,0.080838,0.038877,0.022831,1.702812,2.079319,2022,Sinopec
112,600028,0.073067,0.034562,0.021806,1.584969,2.114081,2023,Sinopec
117,600028,0.058944,0.027604,0.018717,1.474772,2.135395,2024,Sinopec
189,601857,0.081389,0.045828,0.043868,1.044681,1.775949,2021,PetrolChina
194,601857,0.10661,0.061328,0.050623,1.211469,1.738343,2022,PetrolChina
199,601857,0.110566,0.065496,0.059877,1.093836,1.688136,2023,PetrolChina
204,601857,0.107463,0.066744,0.062542,1.06719,1.610075,2024,PetrolChina


In [21]:
# Access the 'roe_markdown' item from the result dictionary.
# It is a plain-text Markdown table ready to paste into an AI chatbot.
# print() displays it as raw text (without extra quoting or escape characters).
print(result["roe_markdown"])

| ROE         | 2021   | 2022   | 2023   | 2024   |
|:------------|:-------|:-------|:-------|:-------|
| PetrolChina | 8.1%   | 10.7%  | 11.1%  | 10.7%  |
| Sinopec     | 9.3%   | 8.1%   | 7.3%   | 5.9%   |


In [22]:
# --- Explore other outputs stored in 'result' ---
# Try accessing any of these keys to see the corresponding output:
#   result['pm_pivot']    → Profit Margin pivot table (as a DataFrame)
#   result['to_pivot']    → Asset Turnover pivot table
#   result['lev_pivot']   → Leverage pivot table
#   result['pm_markdown'] → Profit Margin as a Markdown string (for AI)
#   result['to_markdown'] → Turnover as a Markdown string
#   result['lev_markdown']→ Leverage as a Markdown string
# Example:
# result['pm_pivot']
result['pm_pivot']

,2021,2022,2023,2024
Profit Margin,,,,
PetrolChina,4.4%,5.1%,6.0%,6.3%
Sinopec,3.1%,2.3%,2.2%,1.9%


In [10]:
# Close the WRDS connection opened at the top of the notebook (Part A/B).
# The function in Part D opens and closes its own connection internally,
# so only the top-level 'db' connection needs to be closed here.
#db.close()

In [14]:
result["raw_data"].to_csv("financial_ratios.csv", index=False)
print("The data has been saved as: financial_ratios.csv")

The data has been saved as: financial_ratios.csv
